# Análise de Performance - Verilator vs LUT2GPU

Este notebook processa dados de performance de implementações Verilator e LUT2GPU, usando o nome da pasta como dataset e gerando uma linha para cada arquivo CSV.

## Importando bibliotecas necessárias

In [20]:
import os
import pandas as pd
import numpy as np
from IPython.display import display, HTML

## Função para processar um único arquivo CSV

In [21]:
def process_single_csv(filepath, dataset_name):
    """
    Processa um único arquivo CSV e retorna suas informações.
    
    Args:
        filepath (str): Caminho do arquivo CSV
        dataset_name (str): Nome do dataset (pasta)
    
    Returns:
        pd.DataFrame: DataFrame com uma linha de resultados
    """
    try:
        df = pd.read_csv(filepath)
        
        # Identifica métodos disponíveis
        available_methods = df['implementation'].unique()
        
        # Prepara o DataFrame de resultado com valores padrão
        result_df = pd.DataFrame({
            'Timestamp': [df['timestamp'].min()],
            'Dataset': [dataset_name],
            'Inputs': [df['input_size'].unique()[0]],
            'Outputs': [df['output_size'].unique()[0]],
            'Luts': [df['num_luts'].unique()[0]],
            'Combinations': '2^30',
        })
        
        # Adiciona colunas para cada método
        methods_to_check = ['verilator','cpu', 'cuda', ]

        
        for method in methods_to_check:
            method_data = df[df['implementation'] == method] if method in available_methods else pd.DataFrame()
            
            if not method_data.empty:
                idx_min = method_data['execution_time_ms'].idxmin()
                if method == 'cuda':
                    result_df[f'{method.upper()}_Threads_Block'] = method_data.loc[idx_min, 'threads_per_block']
                    result_df[f'{method.upper()}_Blocks_Grid'] = method_data.loc[idx_min, 'blocks_per_grid']
                result_df[f'{method.upper()}_ExeTime_ms'] = method_data.loc[idx_min, 'execution_time_ms']
                result_df[f'{method.upper()}_Throughput_sec'] = method_data.loc[idx_min, 'throughput_combinations_per_sec']
            else:
                result_df[f'{method.upper()}_ExeTime_ms'] = '-'
                result_df[f'{method.upper()}_Throughput_sec'] = '-'
                if method == 'cuda':
                    result_df[f'{method.upper()}_Threads_Block'] = '-'
                    result_df[f'{method.upper()}_Blocks_Grid'] = '-'
        
        # Calcula diferença se ambos os métodos estiverem presentes e retorna quantas vezes mais rápido é o CUDA
        if 'cpu' in available_methods and 'cuda' in available_methods:
            result_df['CUDA_Speedup_CPU'] = result_df['CUDA_Throughput_sec'] / result_df['CPU_Throughput_sec']
        if 'verilator' in available_methods and 'cuda' in available_methods:
            result_df['CUDA_Speedup_Verilator'] = result_df['CUDA_Throughput_sec'] / result_df['VERILATOR_Throughput_sec']
        else:
            result_df['CUDA_Speedup'] = '-'
        
        return result_df
    
    except Exception as e:
        # print(f"Erro ao processar {filepath}: {e}")
        return None

## Função para processar todas as pastas

In [22]:
def process_all_folders(root_folder, not_got):
    """
    Processa todos os arquivos CSV em todas as pastas.
    
    Args:
        root_folder (str): Pasta raiz para busca
        not_got (list): Lista de pastas para não processar
    
    Returns:
        pd.DataFrame: DataFrame com todos os resultados
    """
    all_results = []
    
    for dirpath, dirnames, filenames in os.walk(root_folder):
        # Pega o nome da última pasta do caminho como dataset
        print(f"Processando: {dirpath}")
        # Se a pasta ou subpasta estiver na lista de não processados, pula
        if any([ng in dirpath for ng in not_got]):
            continue
        
        dataset_name = os.path.basename(dirpath)
        
        for filename in filenames:
            if filename.endswith('.csv'):
                filepath = os.path.join(dirpath, filename)
                result = process_single_csv(filepath, dataset_name)
                
                if result is not None:
                    all_results.append(result)
    
    if not all_results:
        print("Nenhum arquivo CSV encontrado.")
        return None
    
    # Combina todos os resultados
    return pd.concat(all_results, ignore_index=True)

## Função para salvar resultados

In [23]:
def save_results(df, output_path='results/performance_summary.csv'):
    """
    Salva resultados em um arquivo CSV e exibe no notebook
    
    Args:
        df (pd.DataFrame): DataFrame com resultados
        output_path (str): Caminho para salvar o arquivo
    """
    if df is not None:
        df.to_csv(output_path, index=False)
        print(f"Resultados salvos em {output_path}")
        display(HTML(df.to_html(index=False)))

## Processando os dados

Agora vamos processar todos os arquivos CSV encontrados e gerar o relatório.

In [24]:
# Processa os arquivos em todas as pastas
root_folder = 'results'
not_got = ['results_until_26_02']
results = process_all_folders(root_folder, not_got)

if results is not None:
    save_results(results)

Processando: results
Processando: results/SUSY_treelut
Processando: results/SUSY_treelut/verilog
Processando: results/021825_c6288
Processando: results/021825_c6288/cuda
Processando: results/021825_c6288/verilog
Processando: results/021825_c6288/verilator
Processando: results/021825_c6288/verilator/obj_dir
Processando: results/022125_c2670
Processando: results/022125_c2670/verilog
Processando: results/29_03_c880
Processando: results/29_03_c880/cuda
Processando: results/29_03_c880/cpp
Processando: results/29_03_c880/verilog
Processando: results/29_03_c880/verilator
Processando: results/verilog_test
Processando: results/verilog_test/cuda
Processando: results/verilog_test/cpp
Processando: results/verilog_test/verilog
Processando: results/verilog_test/verilator
Processando: results/29_03_c499
Processando: results/29_03_c499/cuda
Processando: results/29_03_c499/cpp
Processando: results/29_03_c499/verilog
Processando: results/29_03_c499/verilator
Processando: results/021825_c499
Processando:

Timestamp,Dataset,Inputs,Outputs,Luts,Combinations,VERILATOR_ExeTime_ms,VERILATOR_Throughput_sec,CPU_ExeTime_ms,CPU_Throughput_sec,CUDA_Threads_Block,CUDA_Blocks_Grid,CUDA_ExeTime_ms,CUDA_Throughput_sec,CUDA_Speedup_CPU,CUDA_Speedup
2025-03-29 14:23:55,29_03_c880,60,26,94,2^30,-,-,1286.0,834946986.0,256.0,768.0,17.342,6.191569e+10,74.155230,-
2025-03-29 14:22:48,29_03_c499,41,32,78,2^30,-,-,879.0,1221549287.83,256.0,768.0,17.192,6.245590e+10,51.128432,-
2025-03-29 14:24:15,29_03_c17,5,2,2,2^30,-,-,97.0,11069503340.209999,256.0,768.0,11.384,9.432026e+10,8.520731,-
2025-03-29 14:24:27,29_03_c6288,32,32,642,2^30,-,-,665.0,1614649359.4,256.0,768.0,34.011,3.157043e+10,19.552498,-
2025-03-29 14:24:06,29_03_c1908,33,25,111,2^30,-,-,667.0,1609807832.08,256.0,768.0,15.638,6.866235e+10,42.652513,-
2025-03-29 14:10:47,rato3,41,32,80,2^30,-,-,-,-,256.0,768.0,17.262,6.220263e+10,NaN,-
2025-03-29 14:22:32,29_03_c5315,178,123,362,2^30,-,-,4018.0,267232907.91,256.0,768.0,47.379,2.266282e+10,84.805505,-
2025-03-29 14:22:58,29_03_c432,36,7,56,2^30,-,-,754.0,1424060774.54,256.0,768.0,13.937,7.704254e+10,54.100596,-
2025-03-29 14:14:05,rato4,178,123,362,2^30,-,-,-,-,256.0,768.0,14.134,7.596872e+10,NaN,-
2025-03-29 14:23:11,29_03_c7552,207,108,443,2^30,-,-,4665.0,230169737.19,256.0,768.0,54.036,1.987086e+10,86.331335,-


In [25]:
if results is not None:
    # Mostra todas as colunas
    pd.set_option('display.max_columns', None)
    display(results)

,Timestamp,Dataset,Inputs,Outputs,Luts,Combinations,VERILATOR_ExeTime_ms,VERILATOR_Throughput_sec,CPU_ExeTime_ms,CPU_Throughput_sec,CUDA_Threads_Block,CUDA_Blocks_Grid,CUDA_ExeTime_ms,CUDA_Throughput_sec,CUDA_Speedup_CPU,CUDA_Speedup
0,2025-03-29 14:23:55,29_03_c880,60,26,94,2^30,-,-,1286.0,834946986.0,256.0,768.0,17.342,6.191569e+10,74.155230,-
1,2025-03-29 14:22:48,29_03_c499,41,32,78,2^30,-,-,879.0,1221549287.83,256.0,768.0,17.192,6.245590e+10,51.128432,-
2,2025-03-29 14:24:15,29_03_c17,5,2,2,2^30,-,-,97.0,11069503340.209999,256.0,768.0,11.384,9.432026e+10,8.520731,-
3,2025-03-29 14:24:27,29_03_c6288,32,32,642,2^30,-,-,665.0,1614649359.4,256.0,768.0,34.011,3.157043e+10,19.552498,-
4,2025-03-29 14:24:06,29_03_c1908,33,25,111,2^30,-,-,667.0,1609807832.08,256.0,768.0,15.638,6.866235e+10,42.652513,-
5,2025-03-29 14:10:47,rato3,41,32,80,2^30,-,-,-,-,256.0,768.0,17.262,6.220263e+10,NaN,-
6,2025-03-29 14:22:32,29_03_c5315,178,123,362,2^30,-,-,4018.0,267232907.91,256.0,768.0,47.379,2.266282e+10,84.805505,-
7,2025-03-29 14:22:58,29_03_c432,36,7,56,2^30,-,-,754.0,1424060774.54,256.0,768.0,13.937,7.704254e+10,54.100596,-
8,2025-03-29 14:14:05,rato4,178,123,362,2^30,-,-,-,-,256.0,768.0,14.134,7.596872e+10,NaN,-
9,2025-03-29 14:23:11,29_03_c7552,207,108,443,2^30,-,-,4665.0,230169737.19,256.0,768.0,54.036,1.987086e+10,86.331335,-
